# Отложенный тест · Airfoil и Power Plant

В сетке v1 по валидации выбрали по одной конфигурации каждого семейства на датасет. Файл `heldout_v1.json` фиксирует архитектуры, столбцы и 24 MLflow run_id. У всех моделей один тестовый фолд и одинаковые три пары seed. Каждый чекпоинт оценивается отдельно.

В Jupyter используй окружение проекта с extra `tracking`, корневой `.env` с `MLFLOW_TRACKING_URI` и исходные файлы обучения в `data/`. Выбери **Run All**. Расчёт идёт на CPU, обращения к MLflow только на чтение. Результаты появятся в `results/grid-v1-heldout/`.

In [ ]:
PROJECT_DIR = None  # При запуске вне проекта укажи абсолютный путь к рабочей копии.

In [ ]:
import sys
from pathlib import Path

if PROJECT_DIR is None:
    cwd = Path.cwd().resolve()
    checkout = next(
        (folder for folder in (cwd, *cwd.parents) if (folder / "src/experiments/runner.py").is_file()),
        None,
    )
else:
    checkout = Path(PROJECT_DIR).expanduser()
    if not checkout.is_absolute() or not (checkout / "src/experiments/runner.py").is_file():
        raise ValueError("Укажи в PROJECT_DIR абсолютный путь к корню mgs-kan на сервере.")
    checkout = checkout.resolve()

if checkout is not None and sys.path[0:1] != [str(checkout)]:
    sys.path.insert(0, str(checkout))

try:
    from src.paths import DATA_DIR, ENV_FILE, PROJECT_ROOT
except ModuleNotFoundError as error:
    raise ImportError("Укажи PROJECT_DIR и выполни ячейки заново.") from error

if checkout is not None and PROJECT_ROOT != checkout:
    raise RuntimeError("Ядро уже загрузило src из другой папки. Перезапусти ядро.")
if not (PROJECT_ROOT / "notebooks/baselines/run_series.ipynb").is_file():
    raise RuntimeError("Укажи PROJECT_DIR с рабочей копией mgs-kan и перезапусти ядро.")
print(f"Проект: {PROJECT_ROOT}\nНастройки: {ENV_FILE}\nДанные: {DATA_DIR}")

In [ ]:
import json
import math
from itertools import product

import pandas as pd
import torch
from IPython.display import display

from src.config import ModelSettings, TrackingSettings
from src.data import load_dataset
from src.models import make_model, parameter_count
from src.tracking import MlflowTracker
from src.training import Trainer

selection_path = PROJECT_ROOT / "notebooks/baselines/heldout_v1.json"
selection = json.loads(selection_path.read_text(encoding="utf-8"))
expected = set(product(("airfoil", "power_plant"), ("mlp", "mgs_kan", "deep_mlp", "resnet")))
seed_pairs = {(11, 101), (22, 202), (33, 303)}
if len(selection) != len(expected) or {(c["dataset"], c["model"]) for c in selection} != expected:
    raise ValueError("Expected one frozen configuration per dataset and model")

run_ids = []
for config in selection:
    trials = config["runs"]
    pairs = {(r["init_seed"], r["shuffle_seed"]) for r in trials}
    if config["fold"] != 0 or pairs != seed_pairs or len(trials) != len(seed_pairs):
        raise ValueError(f"Unexpected fold or seed pairs for {config['dataset']} / {config['model']}")
    run_ids.extend(r["run_id"] for r in trials)
if len(set(run_ids)) != len(run_ids):
    raise ValueError("Duplicate checkpoint run_id")

display(pd.DataFrame(selection).drop(columns="runs"))

## Восстановление и проверка

Сначала сверяем исходный файл, порядок признаков, параметры запуска и лучшую эпоху. Повторно считаем RMSE на валидации. Для переноса весов с GPU на CPU допускаем расхождение до 0,01% или 0,0001 в исходных единицах. К тесту переходим после проверки всех 24 весов.

In [ ]:
def restore(tracker, config, trial, data):
    run_id = trial["run_id"]
    run = tracker.client.get_run(run_id)
    params = run.data.params
    tags = run.data.tags
    if run.info.status != "FINISHED" or tags.get("mgs_kan.evaluation") != "validation":
        raise ValueError(f"{run_id}: expected a finished validation run")

    architecture = config["architecture"] | {"features": data.n_features, "init_seed": trial["init_seed"]}
    expected_params = {
        "model": config["model"], "dataset": data.source_file, "dataset_sha256": data.source_sha256,
        "target_column": data.target_column, "fold": data.fold, "shuffle_seed": trial["shuffle_seed"],
        "trainable_parameters": config["parameters"],
        **{f"model.{key}": value for key, value in architecture.items()},
    }
    for key, value in expected_params.items():
        if params.get(key) != str(value):
            raise ValueError(f"{run_id}: {key} differs from frozen selection or local data")
    if json.loads(params["feature_columns"]) != list(data.selected_features):
        raise ValueError(f"{run_id}: feature order differs from training")

    model = make_model(config["model"], ModelSettings(**architecture))
    if parameter_count(model) != config["parameters"]:
        raise ValueError(f"{run_id}: unexpected parameter count")
    trainer = Trainer(model, tracker=tracker)
    epoch = trainer.load_checkpoint(run_id=run_id)
    if epoch != trial["best_epoch"] or epoch != run.data.metrics["best_epoch"]:
        raise ValueError(f"{run_id}: unexpected checkpoint epoch")
    if not math.isclose(run.data.metrics["best_val_rmse"], trial["val_rmse"], rel_tol=1e-10):
        raise ValueError(f"{run_id}: validation score differs from frozen selection")
    return trainer, int(params["batch_size"])

In [ ]:
def scores(trainer, data, partition, batch_size):
    features, targets = partition.tensors
    predictions = torch.cat([trainer.predict(batch).cpu() for batch in features.split(batch_size)])
    actual = data.inverse_target(targets.double())
    predicted = data.inverse_target(predictions.double())
    sse = (actual - predicted).square().sum().item()
    total = (actual - actual.mean()).square().sum().item()
    rmse = math.sqrt(sse / len(actual))
    if not math.isfinite(rmse) or total == 0:
        raise ValueError("Expected finite predictions and a varying regression target")
    return rmse, 1 - sse / total

In [ ]:
def verified_checkpoints(selection, tracker):
    datasets = {}
    verified = []
    for config in selection:
        key = (config["dataset"], config["fold"])
        if key not in datasets:
            datasets[key] = load_dataset(key[0], fold=key[1], root=DATA_DIR)
        data = datasets[key].select_features(tuple(config["feature_columns"]))
        for trial in config["runs"]:
            trainer, batch_size = restore(tracker, config, trial, data)
            rmse, _ = scores(trainer, data, data.val, batch_size)
            if not math.isclose(rmse, trial["val_rmse"], rel_tol=1e-4, abs_tol=1e-4):
                raise ValueError(f"{trial['run_id']}: validation RMSE {rmse} vs {trial['val_rmse']}")
            verified.append((config, trial, data, trainer, batch_size))
    return verified

tracker = MlflowTracker(TrackingSettings(_env_file=ENV_FILE))
verified = verified_checkpoints(selection, tracker)
print(f"Проверено чекпоинтов: {len(verified)}/{len(run_ids)}")

## Сравнение на тесте

RMSE дан в исходных единицах целевой величины. Меньше — лучше. Для R² больше — лучше. Разброс показывает влияние seed на одном разбиении. В столбце `delta_rmse_vs_mlp` отрицательное значение означает меньшую ошибку, чем у компактного MLP на том же фолде и той же паре seed.

In [ ]:
rows = []
for config, trial, data, trainer, batch_size in verified:
    rmse, r2 = scores(trainer, data, data.test, batch_size)
    metadata = {key: value for key, value in config.items() if key not in ("runs", "architecture", "feature_columns")}
    rows.append({
        **metadata, **trial, "feature_columns": json.dumps(config["feature_columns"]),
        "n_test": len(data.test), "test_rmse": rmse, "test_r2": r2,
    })

runs = pd.DataFrame(rows)
keys = ["dataset", "fold", "init_seed", "shuffle_seed"]
baseline = runs.loc[runs.model == "mlp", keys + ["test_rmse"]].rename(columns={"test_rmse": "mlp_rmse"})
runs = runs.merge(baseline, on=keys, validate="many_to_one")
runs["delta_rmse_vs_mlp"] = runs.test_rmse - runs.mlp_rmse

summary = runs.groupby(["dataset", "model", "stage", "feature_set", "parameters"], as_index=False).agg(
    checkpoints=("run_id", "count"), n_test=("n_test", "first"),
    rmse_mean=("test_rmse", "mean"), rmse_sd=("test_rmse", "std"),
    r2_mean=("test_r2", "mean"), r2_sd=("test_r2", "std"),
    delta_rmse_vs_mlp=("delta_rmse_vs_mlp", "mean"),
).sort_values(["dataset", "rmse_mean"])
display(summary.round(4))

In [ ]:
output_dir = PROJECT_ROOT / "results/grid-v1-heldout"
output_dir.mkdir(parents=True, exist_ok=True)
runs.to_csv(output_dir / "runs.csv", index=False)
summary.to_csv(output_dir / "summary.csv", index=False)
print(f"Сохранено {len(runs)} результатов и {len(summary)} строк сравнения в {output_dir}")